<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Observability for TypeSafe Jev JS/TS with Langfuse" sidebarTitle: "TypeSafe (JS/TS)" logo: "/images/integrations/typesafe_icon.png" description: "Trace TypeSafe Jev System One calls from the JavaScript SDK with Langfuse by wrapping systemOne in observe() and exporting spans through LangfuseSpanProcessor." category: "Integrations" -->

# Observability for TypeSafe Jev JS/TS with Langfuse

<a href="https://langfuse.com/integrations/model-providers/typesafe"><img className="inline" alt="Python" src="https://img.shields.io/badge/Python-3776AB?style=flat&logo=python&logoColor=white" /></a> <a href="https://langfuse.com/integrations/model-providers/typesafe-js"><img className="inline" alt="JS/TS" src="https://img.shields.io/badge/JS/TS-d4d4d8?style=flat&logo=javascript&logoColor=white" /></a>

This notebook shows how to trace **TypeSafe** [Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev) System One calls from the [JavaScript SDK](https://docs.typesafe.ai/sdk/javascript) with **Langfuse**. `@typesafe-ai/sdk` has no OpenTelemetry hook, and there is no `@arizeai/openinference-instrumentation-typesafe`. Wrap `systemOne` with Langfuse `observe()` and register `LangfuseSpanProcessor` so the generation actually exports.

> **What is TypeSafe Jev?** [Jev](https://docs.typesafe.ai/introduction) is TypeSafe's System One model. You send state plus typed [Choice](https://docs.typesafe.ai/primitives/choice), [Score](https://docs.typesafe.ai/primitives/score), and [Noul](https://docs.typesafe.ai/primitives/noul) questions; it returns structured answers with probabilities. It does not generate text. Official [Python](https://docs.typesafe.ai/sdk/python) and [JavaScript](https://docs.typesafe.ai/sdk/javascript) SDKs wrap `POST /v1/systemone`.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate LLM applications. Use [Langfuse Cloud](https://langfuse.com/cloud) or [self-host](https://langfuse.com/self-hosting) it.

<!-- STEPS_START -->
## Step 1: Install Dependencies

Install the TypeSafe JavaScript SDK, the Langfuse tracing packages, and the OpenTelemetry Node SDK. Node.js 20 or newer is required.

```bash
npm install @typesafe-ai/sdk @langfuse/tracing @langfuse/otel @opentelemetry/sdk-node
```

## Step 2: Set Up Environment Variables

Get your Langfuse keys from the project settings in [Langfuse Cloud](https://langfuse.com/cloud) or set up [self-hosting](https://langfuse.com/self-hosting). Get a TypeSafe API key from the [TypeSafe console](https://console.typesafe.ai/settings/keys).

```bash
# Get keys from your project settings: https://langfuse.com/cloud
export LANGFUSE_PUBLIC_KEY="pk-lf-..."
export LANGFUSE_SECRET_KEY="sk-lf-..."
export LANGFUSE_BASE_URL="https://cloud.langfuse.com" # 🇪🇺 EU region (API host)
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

export TYPESAFE_API_KEY="sk-..." # https://console.typesafe.ai/settings/keys
```

## Step 3: Initialize OpenTelemetry with Langfuse

`observe()` only creates spans. They do not reach Langfuse until you register [`LangfuseSpanProcessor`](/docs/observability/sdk/overview#setup) on a Node OpenTelemetry SDK and start that SDK **before** any `systemOne` call.

```typescript
// instrumentation.ts
import { NodeSDK } from "@opentelemetry/sdk-node";
import { LangfuseSpanProcessor } from "@langfuse/otel";

export const sdk = new NodeSDK({
  spanProcessors: [new LangfuseSpanProcessor()],
});

sdk.start();
```

Import this file at the top of your app entry point so the processor is registered before you create observations.

## Step 4: Wrap `systemOne` with `observe()`

Wrap `client.systemOne` with [`observe()`](/docs/observability/sdk/instrumentation#observe-wrapper). Use `asType: "generation"` so Langfuse stores the call as a generation. Keep the `SystemOneRequest<Q>` generic so TypeSafe still infers `response.answers` from your questions. `observe()` captures the request and response automatically. Pass `{ asType: "generation" }` to `updateActiveObservation()` after the SDK returns so the generation also gets the resolved model name and token usage.

Ask Jev three questions about one ticket: a yes/no (Noul), a label (Choice), and a rubric (Score). The same shape covers tool routers, compaction gates, and eval verdicts. Pin `jev-1.13.0` when a threshold depends on a specific model version; `jev-latest` moves when TypeSafe ships a new release.

```typescript
// index.ts
import { sdk } from "./instrumentation";
import { observe, updateActiveObservation } from "@langfuse/tracing";
import {
  TypeSafeClient,
  noul,
  choice,
  score,
  type Questions,
  type SystemOneRequest,
} from "@typesafe-ai/sdk";

const client = new TypeSafeClient({ defaultModel: "jev-1.13.0" });

const systemOne = observe(
  async <Q extends Questions>(request: SystemOneRequest<Q>) => {
    const response = await client.systemOne(request);
    updateActiveObservation(
      {
        model: response.model,
        usageDetails: {
          input: response.usage.input_tokens,
          output: response.usage.output_tokens,
        },
      },
      { asType: "generation" },
    );
    return response;
  },
  { name: "typesafe-system-one", asType: "generation" },
);

async function main() {
  const response = await systemOne({
    state: { document: "I was charged twice. Please fix this ASAP." },
    questions: {
      billing: noul("Is this ticket about billing?"),
      tone: choice("What is the customer's tone?", {
        calm: null,
        frustrated: null,
        angry: null,
      }),
      urgency: score("How urgent is this ticket?", [
        "can wait",
        "this week",
        "today",
      ]),
    },
  });

  console.log(response.model);
  console.log(response.answers.billing.noul);
  console.log(response.answers.tone.choice, response.answers.tone.confidence);
  console.log(response.answers.urgency.score, response.answers.urgency.confidence);
}

main().finally(() => sdk.shutdown());
```

Run the script with `npx tsx index.ts`. `sdk.shutdown()` flushes buffered spans. That call is required in short-lived scripts; a long-running server can skip it until process exit.

To write Jev verdicts back onto Langfuse traces as scores, see [Using TypeSafe's Jev for evals](/blog/2026-09-18-using-typesafes-jev-for-evals).

## Step 5: View Traces in Langfuse

After running the example, open [Langfuse Cloud](https://langfuse.com/cloud) to see the System One generation: request `state` and questions, typed answers with probabilities, token usage, and latency.

![TypeSafe Jev System One trace from the JavaScript SDK in Langfuse](https://langfuse.com/images/cookbook/integration-typesafe/typesafe-example-trace-js.png)

[Example trace in Langfuse](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/45498ea8f1e000c6b112960bfbe63f76?observation=c6ac8287d7cc7837&timestamp=2026-09-22T15:13:29.994Z)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->